In [1]:
from pydantic import BaseModel, Field
from typing import Optional
from llama_cloud_services import LlamaExtract
from llama_cloud.core.api_error import ApiError
from llama_cloud import ExtractConfig
from llama_index.core.workflow import (
    Event,
    StartEvent,
    StopEvent,
    Context,
    Workflow,
    step,
)
from llama_index.llms.anthropic import Anthropic
from llama_index.core.prompts import ChatPromptTemplate
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
class RawFinancials(BaseModel):
    revenue: Optional[float] = Field(
        None, description="Extracted revenue (in million USD)"
    )
    operating_income: Optional[float] = Field(
        None, description="Extracted operating income (in million USD)"
    )
    eps: Optional[float] = Field(
        None, description="Extracted earnings per share"
    )


class InitialFinancialDataOutput(BaseModel):
    company_name: str = Field(..., description="Company name")
    ticker: str = Field(..., description="Stock ticker symbol")
    report_date: str = Field(..., description="Date of the report")
    raw_financials: RawFinancials = Field(
        ..., description="Structured raw financial metrics"
    )


class FinancialModelOutput(BaseModel):
    revenue_projection: float = Field(
        ..., description="Projected revenue for next year (million USD)"
    )
    growth_rate: float = Field(
        ..., description="Expected revenue growth rate (%)"
    )
    valuation_estimate: float = Field(
        ..., description="Estimated enterprise value (million USD)"
    )
    summary: str = Field(
        ..., description="Brief summary of the analysis"
    )


class ComparativeAnalysisOutput(BaseModel):
    comparative_analysis: str = Field(
        ..., description="Comparative analysis between companies"
    )
    overall_recommendation: str = Field(
        ..., description="Investment recommendation with rationale"
    )


class FinalEquityResearchMemoOutput(BaseModel):
    company_a_model: FinancialModelOutput
    company_b_model: FinancialModelOutput
    comparative_analysis: ComparativeAnalysisOutput

In [3]:
class DeckAParseEvent(Event):
    deck_content: InitialFinancialDataOutput


class DeckBParseEvent(Event):
    deck_content: InitialFinancialDataOutput


class CompanyAModelEvent(Event):
    model_output: FinancialModelOutput


class CompanyBModelEvent(Event):
    model_output: FinancialModelOutput

In [7]:
class ComparativeAnalysisWorkflow(Workflow):
    def __init__(self, extract_agent, modeling_path: str, llm=None, **kwargs):
        super().__init__(**kwargs)
        self.extract_agent = extract_agent
        self.llm = llm or Anthropic(model="claude-opus-4-5-20251101", max_tokens=4096)
        with open(modeling_path, "r") as f:
            self.modeling_data = f.read()

    async def _parse_deck(self, deck_path) -> InitialFinancialDataOutput:
        result = await self.extract_agent.aextract(deck_path)
        return InitialFinancialDataOutput(**result.data)

    @step
    async def parse_deck_a(self, ctx: Context, ev: StartEvent) -> DeckAParseEvent:
        output = await self._parse_deck(ev.deck_path_a)
        return DeckAParseEvent(deck_content=output)

    @step
    async def parse_deck_b(self, ctx: Context, ev: StartEvent) -> DeckBParseEvent:
        output = await self._parse_deck(ev.deck_path_b)
        return DeckBParseEvent(deck_content=output)

    async def _generate_financial_model(
        self, financial_data: InitialFinancialDataOutput
    ) -> FinancialModelOutput:
        prompt_str = """You are an expert financial analyst.
Using the following raw financial data and modeling assumptions, 
produce a financial model summary.

Raw Financial Data:
{raw_data}

Financial Modeling Assumptions:
{assumptions}

Return your output as JSON with fields:
- revenue_projection (float): Projected revenue for next year in crores INR
- growth_rate (float): Expected revenue growth rate as percentage
- valuation_estimate (float): Estimated enterprise value in crores INR
- summary (str): Brief summary of the analysis
"""
        prompt = ChatPromptTemplate.from_messages([("user", prompt_str)])
        result = await self.llm.astructured_predict(
            FinancialModelOutput,
            prompt,
            raw_data=financial_data.model_dump_json(),
            assumptions=self.modeling_data,
        )
        return result

    @step
    async def refine_model_a(
        self, ctx: Context, ev: DeckAParseEvent
    ) -> CompanyAModelEvent:
        model = await self._generate_financial_model(ev.deck_content)
        return CompanyAModelEvent(model_output=model)

    @step
    async def refine_model_b(
        self, ctx: Context, ev: DeckBParseEvent
    ) -> CompanyBModelEvent:
        model = await self._generate_financial_model(ev.deck_content)
        return CompanyBModelEvent(model_output=model)

    @step
    async def compare_models(
        self, ctx: Context, ev: CompanyAModelEvent | CompanyBModelEvent
    ) -> StopEvent | None:
        events = ctx.collect_events(ev, [CompanyAModelEvent, CompanyBModelEvent])
        if events is None:
            return None

        model_a_ev, model_b_ev = events
        model_a = model_a_ev.model_output
        model_b = model_b_ev.model_output

        prompt_str = """You are an expert investment analyst.
Compare the following financial models and provide an investment recommendation.

Company A Model:
{company_a_model}

Company B Model:
{company_b_model}

Return your output as JSON with fields:
- comparative_analysis (str): Detailed comparison of the two companies
- overall_recommendation (str): Investment recommendation with rationale
"""
        prompt = ChatPromptTemplate.from_messages([("user", prompt_str)])
        analysis = await self.llm.astructured_predict(
            ComparativeAnalysisOutput,
            prompt,
            company_a_model=model_a.model_dump_json(),
            company_b_model=model_b.model_dump_json(),
        )

        final = FinalEquityResearchMemoOutput(
            company_a_model=model_a,
            company_b_model=model_b,
            comparative_analysis=analysis,
        )
        return StopEvent(result={"memo": final})

In [8]:
llama_extract = LlamaExtract()

try:
    existing = llama_extract.get_agent(name="comparative-analysis")
    if existing:
        llama_extract.delete_agent(existing.id)
except ApiError as e:
    if e.status_code != 404:
        raise

extract_agent = llama_extract.create_agent(
    name="comparative-analysis",
    data_schema=InitialFinancialDataOutput,
    config=ExtractConfig(extraction_mode="BALANCED"),
)
print(f"Created extraction agent: {extract_agent.name}")

Created extraction agent: comparative-analysis


In [9]:
workflow = ComparativeAnalysisWorkflow(
    extract_agent=extract_agent,
    modeling_path="modeling_assumptions.txt",
    timeout=300,
)

result = await workflow.run(
    deck_path_a="drreddys.pdf",
    deck_path_b="biocon.pdf",
)

memo = result["memo"]

Uploading files: 100%|██████████| 1/1 [00:04<00:00,  4.03s/it]

Creating extraction jobs: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]

Extracting files: 100%|██████████| 1/1 [00:07<00:00,  7.73s/it]

Extracting files: 100%|██████████| 1/1 [00:21<00:00, 21.65s/it]


In [10]:
print("=== EQUITY RESEARCH MEMO ===")
print(f"\nCompany A (Dr. Reddy's):")
print(f"  Revenue Projection: ₹{memo.company_a_model.revenue_projection:,.0f} Cr")
print(f"  Growth Rate: {memo.company_a_model.growth_rate}%")
print(f"  Valuation: ₹{memo.company_a_model.valuation_estimate:,.0f} Cr")
print(f"\nCompany B (Ford):")
print(f"  Revenue Projection: ₹{memo.company_b_model.revenue_projection:,.0f} Cr")
print(f"  Growth Rate: {memo.company_b_model.growth_rate}%")
print(f"  Valuation: ₹{memo.company_b_model.valuation_estimate:,.0f} Cr")
print(f"\nComparative Analysis:")
print(memo.comparative_analysis.comparative_analysis)
print(f"\nRecommendation:")
print(memo.comparative_analysis.overall_recommendation)

=== EQUITY RESEARCH MEMO ===

Company A (Dr. Reddy's):
  Revenue Projection: ₹225,861 Cr
  Growth Rate: 10.0%
  Valuation: ₹893,023 Cr

Company B (Ford):
  Revenue Projection: ₹189,422 Cr
  Growth Rate: 11.9%
  Valuation: ₹680,000 Cr

Comparative Analysis:
**Scale & Revenue Base:** Dr. Reddy's Laboratories (Company A) demonstrates significantly larger scale with projected revenue of ₹225,861 crores versus Biocon's ₹189,422 crores, indicating a more established market position and diversified revenue streams across US generics, emerging markets, and domestic operations.

**Growth Dynamics:** Biocon edges ahead on growth rate (11.9% vs 10.0%), reflecting its focus on the high-growth biosimilars segment. This premium growth is characteristic of companies transitioning from traditional generics to specialized biologic medicines, though it comes with higher execution risk.

**Profitability & Margins:** Dr. Reddy's exhibits substantially superior operating margins (~20.7% operating margin) c